# 04 – Model search: many models, many inputs

'''Question:** We have tried baselines, Ridge and LightGBM (notebook 02). Do other model types, other sets of inputs
or better settings (found with random search) give a better prediction of expected use?

**Same set-up as notebook 02:** frozen dataset, same inputs (`src/features.py`), same time split, same error measures,
so the results can be compared directly. Main circuit: NW Plugs & Sockets (45 % of all use).

Notebook 02 compares three model types on all 9 circuits. Notebook 04 goes deep on the most important circuit, north-west plugs (45% of all use, and most of the waste), to test many more models, input sets, horizons and training windows. The next step is to check the main conclusion on all 9 circuits.'''

In [7]:
%pip install statsmodels


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # so we can use Emma's code in src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load, short
from src.features import make_features, split_of
from src.models import metrics

pd.set_option("display.width", 160)

# 1. Data: the frozen dataset, COVID lockdown removed (same as notebook 02)
df = load()

# 2. The circuit we predict
T = "NW|Plugs & Sockets W"
y = df[T]

# 3. Emma's two sets of inputs
X_forecast = make_features(df, T, "forecast")   # calendar + weather + past use (at least 24 h old)
X_normal = make_features(df, T, "normal")       # calendar + weather + slow 28-day level, no recent past use

# 4. Same time split as notebook 02
split = split_of(df["timestamp_local"])
usable = y.notna() & X_forecast.notna().all(axis=1) & X_normal.notna().all(axis=1)
train = usable & (split == "train")
val = usable & (split == "val")
test = usable & (split == "test")

print("Hours:", {"train": int(train.sum()), "validation": int(val.sum()), "test": int(test.sum())})
print("Inputs in the forecast set:", list(X_forecast.columns))
print("Inputs in the normal set:  ", list(X_normal.columns))

# 5. Emma's results on validation, the scores to beat
emma = pd.read_csv("../reports/model_comparison.csv")
emma.query("circuit == @T and period == 'val'")[["model", "MAE_kWh", "CV_RMSE_%"]].sort_values("MAE_kWh")

Hours: {'train': 18013, 'validation': 2284, 'test': 1290}
Inputs in the forecast set: ['hour', 'weekday', 'is_working_day', 'is_holiday', 'is_break', 'is_august_closure', 'doy_sin', 'doy_cos', 'wx_temp_c', 'heating_deg', 'cooling_deg', 'wx_solar_wm2', 'wx_humidity_pct', 'lag_24', 'lag_48', 'lag_168', 'lag_336', 'same_hour_mean_7d']
Inputs in the normal set:   ['hour', 'weekday', 'is_working_day', 'is_holiday', 'is_break', 'is_august_closure', 'doy_sin', 'doy_cos', 'wx_temp_c', 'heating_deg', 'cooling_deg', 'wx_solar_wm2', 'wx_humidity_pct', 'level_work_28d']


,model,MAE_kWh,CV_RMSE_%
4,LightGBM_forecast,0.693958,57.471317
3,Ridge_forecast,0.726444,57.646644
1,B2_median_4w,0.847075,72.995844
6,LightGBM_normal,0.848677,67.350316
0,B1_last_week,0.877633,73.532294
5,Ridge_normal,0.906218,71.991242
2,B3_profile,1.360808,109.015252


## 2. Many models on four input sets
Every model is tried on four sets of inputs, from small to large, so we can see what each group of variables adds.
The table shows exactly which variables are in which set (✓ = used, – = not used).


In [9]:
# All candidate inputs in one table (Emma's forecast inputs + the 28-day level from her normal set)
X_all = pd.concat([X_forecast, X_normal[["level_work_28d"]]], axis=1)

CALENDAR = ["hour", "weekday", "is_working_day", "is_holiday", "is_break", "is_august_closure", "doy_sin", "doy_cos"]
WEATHER = ["wx_temp_c", "heating_deg", "cooling_deg", "wx_solar_wm2", "wx_humidity_pct"]
LEVEL = ["level_work_28d"]
PAST = ["lag_24", "lag_48", "lag_168", "lag_336", "same_hour_mean_7d"]

INPUT_SETS = {
    "1 calendar": CALENDAR,
    "2 calendar + weather": CALENDAR + WEATHER,
    "3 normal (+ 28-day level)": CALENDAR + WEATHER + LEVEL,
    "4 forecast (+ past use)": CALENDAR + WEATHER + PAST,
}

overview = pd.DataFrame(
    {name: ["✓" if v in cols else "–" for v in X_all.columns] for name, cols in INPUT_SETS.items()},
    index=X_all.columns)
overview

,1 calendar,2 calendar + weather,3 normal (+ 28-day level),4 forecast (+ past use)
hour,✓,✓,✓,✓
weekday,✓,✓,✓,✓
is_working_day,✓,✓,✓,✓
is_holiday,✓,✓,✓,✓
is_break,✓,✓,✓,✓
is_august_closure,✓,✓,✓,✓
doy_sin,✓,✓,✓,✓
doy_cos,✓,✓,✓,✓
wx_temp_c,–,✓,✓,✓
heating_deg,–,✓,✓,✓


In [10]:
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor

TREE_MODELS = ["Decision tree", "Random forest", "LightGBM"]

def prepare(cols, model_name):
    """Trees use the inputs as they are. Linear models and the neural network need
    hour/weekday as 0/1 columns and all numbers on the same scale."""
    if model_name in TREE_MODELS:
        return "passthrough"
    cat = [c for c in ["hour", "weekday"] if c in cols]
    num = [c for c in cols if c not in cat]
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                              ("num", StandardScaler(), num)])

# model: (the model, the settings random search may choose from)
MODELS = {
    "Linear regression": (LinearRegression(), {}),
    "Ridge": (Ridge(), {"model__alpha": loguniform(0.01, 1000)}),
    "Lasso": (Lasso(max_iter=5000), {"model__alpha": loguniform(0.1, 100)}),
    "Decision tree": (DecisionTreeRegressor(random_state=42),
                      {"model__max_depth": randint(3, 20), "model__min_samples_leaf": randint(5, 200)}),
    "Random forest": (RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42),
                      {"model__max_depth": randint(5, 25), "model__min_samples_leaf": randint(5, 100),
                       "model__max_features": uniform(0.3, 0.7)}),
    "LightGBM": (LGBMRegressor(objective="regression_l1", subsample_freq=1, random_state=42, verbose=-1, n_jobs=-1),
                 {"model__num_leaves": randint(8, 128), "model__learning_rate": loguniform(0.01, 0.2),
                  "model__n_estimators": randint(200, 1200), "model__min_child_samples": randint(10, 200),
                  "model__subsample": uniform(0.6, 0.4), "model__colsample_bytree": uniform(0.5, 0.5)}),
    "Neural network": (MLPRegressor(max_iter=200, early_stopping=True, random_state=42),
                       {"model__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],
                        "model__alpha": loguniform(1e-5, 1e-1), "model__learning_rate_init": loguniform(1e-4, 1e-2)}),
}

In [13]:
import time
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

results = []      # one row per model; all results in this notebook are collected here

TRIALS = 15        # random settings tried per model
TRIALS_SLOW = 6    # the neural network is slow, so fewer trials

rows = train | val
fold = np.where(val[rows], 0, -1)      # -1 = always train, 0 = the validation period

for set_name, cols in INPUT_SETS.items():
    X, Y = X_all.loc[rows, cols], y[rows]
    for model_name, (estimator, space) in MODELS.items():
        start = time.time()
        pipe = Pipeline([("prepare", prepare(cols, model_name)), ("model", estimator)])
        n_iter = TRIALS_SLOW if model_name == "Neural network" else TRIALS
        if space:
            search = RandomizedSearchCV(pipe, space, n_iter=n_iter, scoring="neg_mean_absolute_error",
                                        cv=PredefinedSplit(fold), refit=False, random_state=42)
            search.fit(X, Y)
            best = search.best_params_
        else:
            best, n_iter = {}, 1
        final = pipe.set_params(**best).fit(X_all.loc[train, cols], y[train])
        score = metrics(y[val], final.predict(X_all.loc[val, cols]))
        results.append({"model": model_name, "inputs": set_name, "trials": n_iter,
                        "best settings": {k.replace("model__", ""): (round(v, 4) if isinstance(v, float) else v)
                                          for k, v in best.items()},
                        **score})
        print(f"{set_name:28s} {model_name:18s} error {score['MAE_kWh']:.3f}  ({time.time() - start:.0f} s)")

1 calendar                   Linear regression  error 0.946  (1 s)
1 calendar                   Ridge              error 0.946  (0 s)
1 calendar                   Lasso              error 0.947  (1 s)
1 calendar                   Decision tree      error 0.798  (0 s)
1 calendar                   Random forest      error 0.821  (5 s)
1 calendar                   LightGBM           error 0.899  (88 s)


/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached 

1 calendar                   Neural network     error 0.804  (44 s)
2 calendar + weather         Linear regression  error 0.951  (0 s)
2 calendar + weather         Ridge              error 0.951  (0 s)
2 calendar + weather         Lasso              error 0.952  (2 s)
2 calendar + weather         Decision tree      error 0.892  (1 s)
2 calendar + weather         Random forest      error 0.914  (9 s)
2 calendar + weather         LightGBM           error 1.054  (94 s)


/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached 

2 calendar + weather         Neural network     error 0.943  (38 s)
3 normal (+ 28-day level)    Linear regression  error 0.914  (0 s)
3 normal (+ 28-day level)    Ridge              error 0.914  (0 s)
3 normal (+ 28-day level)    Lasso              error 0.915  (1 s)
3 normal (+ 28-day level)    Decision tree      error 0.826  (1 s)
3 normal (+ 28-day level)    Random forest      error 0.818  (12 s)
3 normal (+ 28-day level)    LightGBM           error 0.858  (82 s)


/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached 

3 normal (+ 28-day level)    Neural network     error 0.778  (42 s)
4 forecast (+ past use)      Linear regression  error 0.739  (0 s)
4 forecast (+ past use)      Ridge              error 0.739  (0 s)
4 forecast (+ past use)      Lasso              error 0.739  (1 s)
4 forecast (+ past use)      Decision tree      error 0.642  (2 s)
4 forecast (+ past use)      Random forest      error 0.646  (24 s)
4 forecast (+ past use)      LightGBM           error 0.696  (81 s)


/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached 

4 forecast (+ past use)      Neural network     error 0.683  (41 s)


/Users/admin/Desktop/Madrid Progg/2026G1/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [16]:
table = pd.DataFrame(results)
table = table[table["inputs"].notna()]                                    # only the runs from this section
table = table.drop_duplicates(subset=["model", "inputs"], keep="last")    # keep only the latest run of each

print("Average miss on validation (kWh per hour), lower is better. Emma's best: LightGBM forecast 0.694")
display(table.pivot(index="model", columns="inputs", values="MAE_kWh").round(3))
table.sort_values("MAE_kWh")[["model", "inputs", "MAE_kWh", "CV_RMSE_%", "best settings"]].head(10)

Average miss on validation (kWh per hour), lower is better. Emma's best: LightGBM forecast 0.694


inputs,1 calendar,2 calendar + weather,3 normal (+ 28-day level),4 forecast (+ past use)
model,,,,
Decision tree,0.798,0.892,0.826,0.642
Lasso,0.947,0.952,0.915,0.739
LightGBM,0.899,1.054,0.858,0.696
Linear regression,0.946,0.951,0.914,0.739
Neural network,0.804,0.943,0.778,0.683
Random forest,0.821,0.914,0.818,0.646
Ridge,0.946,0.951,0.914,0.739


,model,inputs,MAE_kWh,CV_RMSE_%,best settings
24,Decision tree,4 forecast (+ past use),0.642463,53.447615,"{'max_depth': 6, 'min_samples_leaf': 108}"
25,Random forest,4 forecast (+ past use),0.645564,52.491017,"{'max_depth': 11, 'max_features': 0.8576, 'min..."
27,Neural network,4 forecast (+ past use),0.683134,55.067647,"{'alpha': 0.0, 'hidden_layer_sizes': (64, 32),..."
26,LightGBM,4 forecast (+ past use),0.695619,56.990194,"{'colsample_bytree': 0.9547, 'learning_rate': ..."
21,Linear regression,4 forecast (+ past use),0.738554,58.128224,{}
22,Ridge,4 forecast (+ past use),0.738554,58.128237,{'alpha': 0.0127}
23,Lasso,4 forecast (+ past use),0.738819,58.158543,{'alpha': 0.1153}
20,Neural network,3 normal (+ 28-day level),0.777799,64.840374,"{'alpha': 0.0, 'hidden_layer_sizes': (64, 32),..."
31,Decision tree,1 calendar,0.797984,68.341866,"{'max_depth': 5, 'min_samples_leaf': 154}"
6,Neural network,1 calendar,0.804301,67.059079,"{'alpha': 0.0002, 'hidden_layer_sizes': (128, ..."


## 3. Walk-forward validation
One validation period can be lucky. Here the best models are tested month by month (Sep 2021 – Dec 2022):
for each month, the model is trained on everything before it and predicts that month. The test period
(Jan–Feb 2023) is still not used.

In [19]:
from src.models import baseline_median_4w

cols = CALENDAR + WEATHER + PAST          # input set 4 (forecast), the best set in section 2

# The best models from section 2, with the settings random search chose (see the "best settings" column)
FINALISTS = {
    "Linear regression": LinearRegression(),
    "Decision tree": DecisionTreeRegressor(max_depth=6, min_samples_leaf=108, random_state=42),
    "Random forest": RandomForestRegressor(n_estimators=100, max_depth=11, min_samples_leaf=19,
                                           max_features=0.8576, n_jobs=-1, random_state=42),
    "LightGBM": LGBMRegressor(objective="regression_l1", num_leaves=15, learning_rate=0.03,
                                     n_estimators=600, min_child_samples=20, subsample=0.8, subsample_freq=1,
                                     colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=-1),
}

b2 = baseline_median_4w(df, T)                           # 4-week median baseline
month = df["timestamp_local"].dt.to_period("M")          # e.g. 2022-03
TEST_MONTHS = pd.period_range("2021-09", "2022-12", freq="M")

wf_rows = []
for m in TEST_MONTHS:
    fit = usable & (month < m) & split.isin(["train", "val"])   # everything BEFORE this month
    score_rows = usable & (month == m)                          # only this month
    if score_rows.sum() < 200:                                  # skip months with too little data
        continue
    wf_rows.append({"month": str(m), "model": "Baseline: 4-week median", **metrics(y[score_rows], b2[score_rows])})
    for name, est in FINALISTS.items():
        pipe = Pipeline([("prepare", prepare(cols, name)), ("model", est)])
        pipe.fit(X_all.loc[fit, cols], y[fit])
        wf_rows.append({"month": str(m), "model": name, **metrics(y[score_rows], pipe.predict(X_all.loc[score_rows, cols]))})
    print(m, "done")

2021-09 done
2021-10 done
2021-11 done
2021-12 done
2022-01 done
2022-02 done
2022-03 done
2022-04 done
2022-05 done
2022-06 done
2022-07 done
2022-08 done
2022-10 done
2022-11 done
2022-12 done


In [20]:
wf = pd.DataFrame(wf_rows)

monthly = wf.pivot(index="month", columns="model", values="MAE_kWh").round(3)
display(monthly)

summary = pd.DataFrame({
    "average error": wf.groupby("model")["MAE_kWh"].mean(),
    "months won": monthly.idxmin(axis=1).value_counts(),
}).fillna(0).sort_values("average error")
summary.round(3)

model,Baseline: 4-week median,Decision tree,LightGBM,Linear regression,Random forest
month,,,,,
2021-09,0.196,0.200,0.177,0.201,0.199
2021-10,0.316,0.314,0.266,0.292,0.286
2021-11,1.391,1.368,1.338,1.122,1.342
2021-12,1.285,0.995,1.007,0.978,0.996
2022-01,1.073,0.774,0.809,0.761,0.790
2022-02,1.041,0.970,1.007,1.002,0.980
2022-03,1.519,1.055,0.940,0.968,1.030
2022-04,0.812,0.769,0.753,0.736,0.704
2022-05,0.355,0.201,0.158,0.231,0.187


,average error,months won
Random forest,0.580,3.0
LightGBM,0.586,8.0
Decision tree,0.594,1.0
Linear regression,0.595,3.0
Baseline: 4-week median,0.725,0.0


## 4. Forecast horizons: 1 hour, 1 day and 1 week ahead
How far ahead we predict decides which past values the model may use: only what is already known when the
prediction is made. The calendar and weather inputs stay the same; only the past-use inputs change.
Each horizon is tested with walk-forward validation (Sep 2021 – Dec 2022), on exactly the same hours.

| Horizon | Use case | Most recent past value allowed |
|---|---|---|
| 1 hour ahead | live alarm | 1 hour ago |
| 1 day ahead | daily report (our current set-up) | 24 hours ago |
| 1 week ahead | weekly planning and budget | 7 days ago |

In [21]:
from src.features import _per_regime

def shifted(hours):
    """Use of the same circuit `hours` earlier, computed inside each period (never across the 2.5-year gap)."""
    return _per_regime(df, y, lambda s: s.shift(hours))

def same_hour_mean(first_lag, n, step):
    """Mean of n past values at the same hour: first_lag, first_lag + step, ... hours earlier."""
    return _per_regime(df, y, lambda s: pd.concat([s.shift(first_lag + step * k) for k in range(n)], axis=1).mean(axis=1))

# Past-use inputs allowed for each horizon
HORIZONS = {
    "1 hour ahead": {
        "lag_1": shifted(1), "lag_2": shifted(2), "lag_24": shifted(24), "lag_168": shifted(168),
        "mean_last_24h": _per_regime(df, y, lambda s: s.shift(1).rolling(24, min_periods=12).mean()),
        "same_hour_mean_7d": same_hour_mean(24, 7, 24)},
    "1 day ahead": {
        "lag_24": shifted(24), "lag_48": shifted(48), "lag_168": shifted(168), "lag_336": shifted(336),
        "same_hour_mean_7d": same_hour_mean(24, 7, 24)},
    "1 week ahead": {
        "lag_168": shifted(168), "lag_336": shifted(336), "lag_504": shifted(504),
        "same_hour_mean_4w": same_hour_mean(168, 4, 168)},
}
for h, past in HORIZONS.items():
    print(f"{h:13s}", list(past))

# Calendar + weather (same for all) + the past-use inputs of each horizon
X_h = {h: pd.concat([X_forecast[CALENDAR + WEATHER], pd.DataFrame(past)], axis=1) for h, past in HORIZONS.items()}

# Only hours where every horizon has all its inputs, so all three are scored on the same hours
usable_h = usable.copy()
for X in X_h.values():
    usable_h &= X.notna().all(axis=1)
print("Usable hours in all horizons:", int(usable_h.sum()), "of", int(usable.sum()))

1 hour ahead  ['lag_1', 'lag_2', 'lag_24', 'lag_168', 'mean_last_24h', 'same_hour_mean_7d']
1 day ahead   ['lag_24', 'lag_48', 'lag_168', 'lag_336', 'same_hour_mean_7d']
1 week ahead  ['lag_168', 'lag_336', 'lag_504', 'same_hour_mean_4w']
Usable hours in all horizons: 21800 of 22392


In [24]:
HZ_MODELS = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(n_estimators=100, max_depth=11, min_samples_leaf=19,
                                           max_features=0.8576, n_jobs=-1, random_state=42),
    "LightGBM": LGBMRegressor(objective="regression_l1", num_leaves=15, learning_rate=0.03,
                                     n_estimators=600, min_child_samples=20, subsample=0.8, subsample_freq=1,
                                     colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=-1),
}

hz_rows = []
for h, X in X_h.items():
    cols = list(X.columns)
    for m in TEST_MONTHS:
        fit = usable_h & (month < m) & split.isin(["train", "val"])
        score_rows = usable_h & (month == m)
        if score_rows.sum() < 200:
            continue
        for name, est in HZ_MODELS.items():
            pipe = Pipeline([("prepare", prepare(cols, name)), ("model", est)]).fit(X.loc[fit], y[fit])
            hz_rows.append({"horizon": h, "month": str(m), "model": name,
                            **metrics(y[score_rows], pipe.predict(X.loc[score_rows]))})
    print(h, "done")

1 hour ahead done
1 day ahead done
1 week ahead done


In [23]:
hz = pd.DataFrame(hz_rows)
order = ["1 hour ahead", "1 day ahead", "1 week ahead"]

print("Average error over 15 months (kWh per hour), lower is better")
display(hz.pivot_table(index="model", columns="horizon", values="MAE_kWh", aggfunc="mean")[order].round(3))

print("Error as % of average use (CV(RMSE)), ASHRAE target: under 30 %")
hz.pivot_table(index="model", columns="horizon", values="CV_RMSE_%", aggfunc="mean")[order].round(1)

Average error over 15 months (kWh per hour), lower is better


horizon,1 hour ahead,1 day ahead,1 week ahead
model,,,
LightGBM (Emma),0.287,0.583,0.721
Linear regression,0.303,0.592,0.687
Random forest,0.303,0.578,0.687


Error as % of average use (CV(RMSE)), ASHRAE target: under 30 %


horizon,1 hour ahead,1 day ahead,1 week ahead
model,,,
LightGBM (Emma),35.4,56.6,64.7
Linear regression,34.9,54.4,62.2
Random forest,35.0,54.4,62.5


## 5. Training window: all history or only recent data?
Use grows from year to year (notebook 01), so old data might describe a building that no longer exists.
We test, with walk-forward validation, whether a model learns better from all history, the last 12 months
or the last 6 months. Inputs: set 4 (forecast, 1 day ahead).


In [25]:
cols = CALENDAR + WEATHER + PAST                 # input set 4, 1 day ahead
WINDOWS = {"all history": None, "last 12 months": 12, "last 6 months": 6}

tw_rows = []
for window_name, n_months in WINDOWS.items():
    for m in TEST_MONTHS:
        fit = usable & (month < m) & split.isin(["train", "val"])    # everything before this month ...
        if n_months is not None:
            fit &= month >= m - n_months                             # ... but only the last n months
        score_rows = usable & (month == m)
        if score_rows.sum() < 200:
            continue
        for name, est in HZ_MODELS.items():
            pipe = Pipeline([("prepare", prepare(cols, name)), ("model", est)])
            pipe.fit(X_all.loc[fit, cols], y[fit])
            tw_rows.append({"window": window_name, "month": str(m), "model": name,
                            "training hours": int(fit.sum()),
                            **metrics(y[score_rows], pipe.predict(X_all.loc[score_rows, cols]))})
    print(window_name, "done")

all history done
last 12 months done
last 6 months done


In [26]:
tw = pd.DataFrame(tw_rows)
order = list(WINDOWS)

print("Average error over 15 months (kWh per hour), lower is better")
display(tw.pivot_table(index="model", columns="window", values="MAE_kWh", aggfunc="mean")[order].round(3))

# Split into winter (heating season) and the rest of the year
tw["season"] = np.where(pd.PeriodIndex(tw["month"], freq="M").month.isin([11, 12, 1, 2, 3]),
                        "winter (Nov–Mar)", "rest of year")
print("Same, per season")
display(tw.pivot_table(index=["season", "model"], columns="window", values="MAE_kWh", aggfunc="mean")[order].round(3))

print("Average number of training hours per window")
tw.groupby("window")["training hours"].mean().round(0)

Average error over 15 months (kWh per hour), lower is better


window,all history,last 12 months,last 6 months
model,,,
LightGBM,0.586,0.589,0.685
Linear regression,0.595,0.612,0.747
Random forest,0.580,0.589,0.703


Same, per season


window                              all history  last 12 months  last 6 months
season           model                                                        
rest of year     LightGBM                 0.237           0.241          0.286
                 Linear regression        0.270           0.302          0.433
                 Random forest            0.242           0.251          0.334
winter (Nov–Mar) LightGBM                 0.985           0.986          1.141
                 Linear regression        0.967           0.967          1.106
                 Random forest            0.966           0.974          1.124

Average number of training hours per window


window
all history       14652.0
last 12 months     8314.0
last 6 months      4054.0
Name: training hours, dtype: float64